In [ ]:
import requests
import json
header = {'User-Agent':'Mozilla/5.0 (Linux; Android 6.0; Nexus 5 Build/MRA58N) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/139.0.0.0 Mobile Safari/537.36'}

response = requests.get("https://image.baidu.com/search/acjson?tn=resultjson_com&logid=3345384492595205401&ipn=rj&"
                        "ct=201326592&is=&fp=result&fr=ala&word=%E7%8C%AB%E5%92%AA%E5%9B%BE%E7%89%87&queryWord="
                        "%E7%8C%AB%E5%92%AA%E5%9B%BE%E7%89%87&cl=2&lm=-1&ie=utf-8&oe=utf-8&adpicid=&st=&z=&ic=&hd=&latest=&copyright="
                        "&s=&se=&tab=&width=&height=&face=&istype=&qc=&nc=&expermode=&nojc=&isAsync=&pn=100&rn=100&gsm=3c&1662186069094=",
                        headers=header)

# response = requests.get("https://image.baidu.com/search/acjson?tn=resultjson_com&word=%E5%B0%8F%E7%8B%97&ie=utf-8&fp=result&fr=&ala=0&applid=9216922044399147381&pn=30&rn=30&nojc=0&gsm=1e&newReq=1"
#                         # "ct=201326592&is=&fp=result&fr=ala&word=%E7%8C%AB%E5%92%AA%E5%9B%BE%E7%89%87&queryWord="
#                         "ct=201326592&is=&fp=result&fr=ala&word=%E5%B0%8F%E7%8B%97%E5%9B%BE%E7%89%87&queryWord="
#                         "%E5%B0%8F%E7%8B%97%E5%9B%BE%E7%89%87&cl=2&lm=-1&ie=utf-8&oe=utf-8&adpicid=&st=&z=&ic=&hd=&latest=&copyright="
#                         "&s=&se=&tab=&width=&height=&face=&istype=&qc=&nc=&expermode=&nojc=&isAsync=&pn=10&rn=200&gsm=3c&1662186069094=",
#                         headers=header)


data = response.text
obj = json.loads(data)
starList = obj['data']

x = 1
for item in starList:
    if 'thumbURL' in item:
        print(item['thumbURL'])
        responseIMG = requests.get(item['thumbURL'])
        content = responseIMG.content   # 获取图片内容,二进制流
        with open('./img/' + str(x) + '.jpg', 'wb') as f:
            f.write(content)
        x = x + 1

In [2]:
import requests
import json
import time
import random
import os
from urllib.parse import quote

def baidu_image_spider(keyword, download_num, save_dir='./img'):
    """
    百度图片爬虫函数
    :param keyword: 搜索关键词，如 '猫咪'
    :param download_num: 想要下载的图片数量
    :param save_dir: 图片保存目录
    """
    
    # 创建保存目录
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
    
    # 请求头
    header = {
        'User-Agent': 'Mozilla/5.0 (Linux; Android 6.0; Nexus 5 Build/MRA58N) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/139.0.0.0 Mobile Safari/537.36'
    }
    
    # 用于去重的集合
    downloaded_urls = set()
    downloaded_count = 0
    page_size = 60  # 百度每页返回的大概数量
    pn = 0  # 起始页码
    
    while downloaded_count < download_num:
        # 编码关键词
        encoded_keyword = quote(keyword)
        
        # 修复URL格式 - 使用能成功获取数据的格式
        url = f"https://image.baidu.com/search/acjson?tn=resultjson_com&logid=3345384492595205401&ipn=rj&" \
              f"ct=201326592&is=&fp=result&fr=ala&word={encoded_keyword}&queryWord=" \
              f"{encoded_keyword}&cl=2&lm=-1&ie=utf-8&oe=utf-8&adpicid=&st=&z=&ic=&hd=&latest=&copyright=" \
              f"&s=&se=&tab=&width=&height=&face=&istype=&qc=&nc=&expermode=&nojc=&isAsync=&pn={pn}&rn={page_size}&gsm=3c&{int(time.time()*1000)}="
        
        try:
            print(f"正在请求第 {pn//page_size + 1} 页...")
            response = requests.get(url, headers=header, timeout=10)
            response.encoding = 'utf-8'
            
            # 处理JSON数据
            data = response.text
            
            try:
                obj = json.loads(data)
                image_list = obj.get('data', [])
                
                if not image_list:
                    print("没有更多图片了")
                    break
                
                print(f"本页获取到 {len(image_list)} 张图片")
                
                for item in image_list:
                    if downloaded_count >= download_num:
                        break
                    
                    # 获取图片URL（优先使用thumbURL）
                    thumbURL = item.get('thumbURL') or item.get('middleURL') or item.get('objURL')
                    
                    if thumbURL and thumbURL not in downloaded_urls:
                        downloaded_urls.add(thumbURL)
                        
                        try:
                            # 下载图片
                            print(f"正在下载第 {downloaded_count + 1} 张图片...")
                            responseIMG = requests.get(thumbURL, headers=header, timeout=15)
                            
                            if responseIMG.status_code == 200:
                                # 生成唯一文件名
                                file_ext = '.jpg'  # 默认扩展名
                                if 'webp' in thumbURL.lower():
                                    file_ext = '.webp'
                                elif 'png' in thumbURL.lower():
                                    file_ext = '.png'
                                elif 'gif' in thumbURL.lower():
                                    file_ext = '.gif'
                                
                                filename = f"{downloaded_count + 1}{file_ext}"
                                filepath = os.path.join(save_dir, filename)
                                
                                with open(filepath, 'wb') as f:
                                    f.write(responseIMG.content)
                                
                                downloaded_count += 1
                                
                                
                                # 随机延迟，避免请求过快
                                # time.sleep(random.uniform(0.5, 1.5))
                                
                            else:
                                print(f"下载失败，状态码: {responseIMG.status_code}")
                                
                        except Exception as e:
                            print(f"下载图片时出错: {e}")
                            continue
                
            except json.JSONDecodeError as e:
                print(f"JSON解析错误: {e}")
                print(f"原始数据: {data[:200]}...")
                break
                
        except Exception as e:
            print(f"请求错误: {e}")
            break
        
        # 翻到下一页
        pn += page_size
        
        # 页面间延迟
        time.sleep(random.uniform(1, 2))
    
    print(f"\n下载完成！共下载 {downloaded_count} 张图片到目录: {save_dir}")

# 使用示例
if __name__ == "__main__":
    # 在这里设置你的参数
    search_keyword = "猫咪图片"  # 搜索关键词 - 修改为中文而非编码
    desired_count = 100     # 想要下载的图片数量 - 先测试少量
    save_directory = "./OpenCV_dataset"  # 保存目录

    # 开始爬取
    baidu_image_spider(search_keyword, desired_count, save_directory)

正在请求第 1 页...
本页获取到 61 张图片
正在下载第 1 张图片...
正在下载第 2 张图片...
正在下载第 3 张图片...
正在下载第 4 张图片...
正在下载第 5 张图片...
正在下载第 6 张图片...
正在下载第 7 张图片...
正在下载第 8 张图片...
正在下载第 9 张图片...
正在下载第 10 张图片...
正在下载第 11 张图片...
正在下载第 12 张图片...
正在下载第 13 张图片...
正在下载第 14 张图片...
正在下载第 15 张图片...
正在下载第 16 张图片...
正在下载第 17 张图片...
正在下载第 18 张图片...
正在下载第 19 张图片...
正在下载第 20 张图片...
正在下载第 21 张图片...
正在下载第 22 张图片...
正在下载第 23 张图片...
正在下载第 24 张图片...
正在下载第 25 张图片...
正在下载第 26 张图片...
正在下载第 27 张图片...
正在下载第 28 张图片...
正在下载第 29 张图片...
正在下载第 30 张图片...
正在下载第 31 张图片...
正在下载第 32 张图片...
正在下载第 33 张图片...
正在下载第 34 张图片...
正在下载第 35 张图片...
正在下载第 36 张图片...
正在下载第 37 张图片...
正在下载第 38 张图片...
正在下载第 39 张图片...
正在下载第 40 张图片...
正在下载第 41 张图片...
正在下载第 42 张图片...
正在下载第 43 张图片...
正在下载第 44 张图片...
正在下载第 45 张图片...
正在下载第 46 张图片...
正在下载第 47 张图片...
正在下载第 48 张图片...
正在下载第 49 张图片...
正在下载第 50 张图片...
正在下载第 51 张图片...
正在下载第 52 张图片...
正在下载第 53 张图片...
正在下载第 54 张图片...
正在下载第 55 张图片...
正在下载第 56 张图片...
正在下载第 57 张图片...
正在下载第 58 张图片...
正在下载第 59 张图片...
正在下载第 60 张图片...
正在请求第 2 页...
本页获取到 61 张